In [1]:
import pandas as pd
import numpy as np

RANDOM_STATE = 42

In [2]:
df_train = pd.read_csv(f"data-00-raw/bike_train.csv", index_col=[0], parse_dates=["date"])

df_train

,date,wcond,temp,atemp,hum,wind,count
1,2022-01-01,2,9.1,13.2,79.78,10.43,982
2,2022-01-02,2,9.9,12.7,68.91,16.16,797
3,2022-01-03,1,3.1,4.5,43.29,16.14,1351
4,2022-01-04,1,3.2,5.6,58.45,10.42,1559
5,2022-01-05,1,4.3,6.5,43.26,12.15,1597
...,...,...,...,...,...,...,...
361,2022-12-27,2,8.3,11.4,75.49,12.25,1161
362,2022-12-28,1,7.3,9.0,49.89,19.11,2301
363,2022-12-29,1,5.2,8.2,56.84,7.76,2421
364,2022-12-30,1,7.8,10.9,63.03,8.73,2995


In [3]:
df_test = pd.read_csv(f"data-00-raw/bike_test.csv", index_col=[0], parse_dates=["date"])

df_test

,date,wcond,temp,atemp,hum,wind
1,2023-01-01,1,10.2,13.8,68.56,12.49
2,2023-01-02,1,6.2,7.6,37.75,21.43
3,2023-01-03,1,1.1,1.3,43.68,23.77
4,2023-01-04,2,-0.6,1.0,41.04,12.01
5,2023-01-05,1,5.9,8.9,51.89,8.45
...,...,...,...,...,...,...
361,2023-12-27,3,5.0,6.0,81.51,20.58
362,2023-12-28,2,5.4,6.3,64.64,22.76
363,2023-12-29,2,5.4,7.8,58.41,10.11
364,2023-12-30,2,5.4,7.1,74.54,8.08


# Features Engineering

In [4]:
import holidays

In [5]:
def add_fourier_features(df: pd.DataFrame, n_harmonics: int = 3) -> pd.DataFrame:
    """Add Fourier seasonality terms for annual cycle.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with ``dayofyear`` column.
    n_harmonics : int
        Number of harmonics to generate.

    Returns
    -------
    pd.DataFrame
        DataFrame with added sin/cos Fourier columns.
    """
    for k in range(1, n_harmonics + 1):
        df[f"sin_doy_{k}"] = np.sin(2 * np.pi * k * df["dayofyear"] / 365)
        df[f"cos_doy_{k}"] = np.cos(2 * np.pi * k * df["dayofyear"] / 365)
    return df

def build_features(
        df: pd.DataFrame,
        to_drop_at_end: list[str]|None = None
    ) -> pd.DataFrame:
    """Build full feature matrix from raw bike sharing data.

    Parameters
    ----------
    df : pd.DataFrame
        Raw DataFrame with columns: date, temp, atemp, hum, wind, wcond.

    Returns
    -------
    pd.DataFrame
        DataFrame with engineered features, without target column.
    """
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    df["month"] = df["date"].dt.month
    df["dayofweek"] = df["date"].dt.dayofweek
    df["dayofyear"] = df["date"].dt.dayofyear
    df["quarter"] = df["date"].dt.quarter
    df["weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["doy_normalized"] = df["dayofyear"] / 365

    df["t"] = (df["date"] - pd.Timestamp("2022-01-01")).dt.days

    df["temp_sq"] = df["temp"] ** 2
    df["temp_cube"] = df["temp"] ** 3
    df["hum_sq"] = df["hum"] ** 2
    df["wind_sq"] = df["wind"] ** 2
    df["temp_atemp_diff"] = df["temp"] - df["atemp"]

    df["temp_x_hum"] = df["temp"] * df["hum"]
    df["temp_x_wind"] = df["temp"] * df["wind"]
    df["atemp_x_hum"] = df["atemp"] * df["hum"]
    df["hum_x_wind"] = df["hum"] * df["wind"]
    df["wcond_x_temp"] = df["wcond"] * df["temp"]

    # New Features
    ##############
    df["is_summer"] = df["month"].isin([6, 7, 8]).astype(int)
    df["is_winter"] = df["month"].isin([12, 1, 2]).astype(int)


    df["is_summer_break_schools"] = df["month"].isin([6, 7, 8]).astype(int)
    df["is_summer_break_academia"] = df["month"].isin([7, 8, 9]).astype(int)


    checker_holidays = holidays.Poland(years=df["date"].dt.year)
    df["is_holiday"] = df["date"].isin(checker_holidays).astype(int)
    
    # END New Features
    ##################

    wcond_dummies = pd.get_dummies(df["wcond"], prefix="wcond", drop_first=True)
    df = pd.concat([df, wcond_dummies], axis=1)

    df = add_fourier_features(df)

    drop_cols = ["date", "wcond", "dayofyear"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    if to_drop_at_end is not None:
        df = df.drop(columns=to_drop_at_end, errors='ignore')
    return df

In [6]:
df_train_featured = df_train.copy()


df_train_featured = build_features(
    df_train_featured,
)

df_train_featured

,temp,atemp,hum,wind,count,month,dayofweek,quarter,weekend,doy_normalized,...,is_summer_break_academia,is_holiday,wcond_2,wcond_3,sin_doy_1,cos_doy_1,sin_doy_2,cos_doy_2,sin_doy_3,cos_doy_3
1,9.1,13.2,79.78,10.43,982,1,5,1,1,0.002740,...,0,0,True,False,1.721336e-02,0.999852,3.442161e-02,0.999407,5.161967e-02,0.998667
2,9.9,12.7,68.91,16.16,797,1,6,1,1,0.005479,...,0,0,True,False,3.442161e-02,0.999407,6.880243e-02,0.997630,1.031017e-01,0.994671
3,3.1,4.5,43.29,16.14,1351,1,0,1,0,0.008219,...,0,0,False,False,5.161967e-02,0.998667,1.031017e-01,0.994671,1.543088e-01,0.988023
4,3.2,5.6,58.45,10.42,1559,1,1,1,0,0.010959,...,0,0,False,False,6.880243e-02,0.997630,1.372788e-01,0.990532,2.051045e-01,0.978740
5,4.3,6.5,43.26,12.15,1597,1,2,1,0,0.013699,...,0,0,False,False,8.596480e-02,0.996298,1.712931e-01,0.985220,2.553533e-01,0.966848
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,8.3,11.4,75.49,12.25,1161,12,1,4,0,0.989041,...,0,0,True,False,-6.880243e-02,0.997630,-1.372788e-01,0.990532,-2.051045e-01,0.978740
362,7.3,9.0,49.89,19.11,2301,12,2,4,0,0.991781,...,0,0,False,False,-5.161967e-02,0.998667,-1.031017e-01,0.994671,-1.543088e-01,0.988023
363,5.2,8.2,56.84,7.76,2421,12,3,4,0,0.994521,...,0,0,False,False,-3.442161e-02,0.999407,-6.880243e-02,0.997630,-1.031017e-01,0.994671
364,7.8,10.9,63.03,8.73,2995,12,4,4,0,0.997260,...,0,0,False,False,-1.721336e-02,0.999852,-3.442161e-02,0.999407,-5.161967e-02,0.998667


In [7]:
df_test_featured = df_test.copy()

df_test_featured = build_features(
    df_test_featured, 
)

df_test_featured

,temp,atemp,hum,wind,month,dayofweek,quarter,weekend,doy_normalized,t,...,is_summer_break_academia,is_holiday,wcond_2,wcond_3,sin_doy_1,cos_doy_1,sin_doy_2,cos_doy_2,sin_doy_3,cos_doy_3
1,10.2,13.8,68.56,12.49,1,6,1,1,0.002740,365,...,0,0,False,False,1.721336e-02,0.999852,3.442161e-02,0.999407,5.161967e-02,0.998667
2,6.2,7.6,37.75,21.43,1,0,1,0,0.005479,366,...,0,0,False,False,3.442161e-02,0.999407,6.880243e-02,0.997630,1.031017e-01,0.994671
3,1.1,1.3,43.68,23.77,1,1,1,0,0.008219,367,...,0,0,False,False,5.161967e-02,0.998667,1.031017e-01,0.994671,1.543088e-01,0.988023
4,-0.6,1.0,41.04,12.01,1,2,1,0,0.010959,368,...,0,0,True,False,6.880243e-02,0.997630,1.372788e-01,0.990532,2.051045e-01,0.978740
5,5.9,8.9,51.89,8.45,1,3,1,0,0.013699,369,...,0,0,False,False,8.596480e-02,0.996298,1.712931e-01,0.985220,2.553533e-01,0.966848
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,5.0,6.0,81.51,20.58,12,2,4,0,0.989041,725,...,0,0,False,True,-6.880243e-02,0.997630,-1.372788e-01,0.990532,-2.051045e-01,0.978740
362,5.4,6.3,64.64,22.76,12,3,4,0,0.991781,726,...,0,0,True,False,-5.161967e-02,0.998667,-1.031017e-01,0.994671,-1.543088e-01,0.988023
363,5.4,7.8,58.41,10.11,12,4,4,0,0.994521,727,...,0,0,True,False,-3.442161e-02,0.999407,-6.880243e-02,0.997630,-1.031017e-01,0.994671
364,5.4,7.1,74.54,8.08,12,5,4,1,0.997260,728,...,0,0,True,False,-1.721336e-02,0.999852,-3.442161e-02,0.999407,-5.161967e-02,0.998667


### X_[train|test] + y_[train] 

In [8]:
X = df_train_featured.drop(columns=["count"]).to_numpy()

y_sqrt = np.sqrt(df_train_featured["count"].to_numpy())

y_log = np.log1p(df_train_featured["count"].to_numpy())

In [9]:
X_test = df_test_featured.to_numpy()

# Xgboost + Optuna

In [10]:
import xgboost as xgb
import optuna

from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.model_selection import GroupKFold

In [11]:
kf = GroupKFold(n_splits=6)

groups: np.ndarray = df_train["date"].dt.isocalendar().week.astype(int).to_numpy()

In [12]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),

        "subsample": trial.suggest_float("subsample", 0.4, 0.8), 
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.8),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4, 0.8),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.4, 0.8),

        "min_child_weight": trial.suggest_int("min_child_weight", 5, 30), 
        "gamma": trial.suggest_float("gamma", 0.0, 5.0), 
        "max_delta_step": trial.suggest_float("max_delta_step", 0.0, 5.0), 

        "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 50.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 50.0, log=True),
        
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
    }
    val_rmses = []
    for train_idx, val_idx in kf.split(X, y_log, groups=groups):
        model = xgb.XGBRegressor(**params)

        model.fit(X[train_idx], y_log[train_idx])
        val_pred = np.expm1(model.predict(X[val_idx]))
        y_val_orig = np.expm1(y_log[val_idx])
        val_rmses.append(rmse(y_val_orig, val_pred))
    return float(np.mean(val_rmses))

In [13]:
study = optuna.create_study(
    study_name="xgb_bike",
    storage="sqlite:///params-optuna/optuna_xgb_new_features_I1.db", 
    direction="minimize",
    load_if_exists=True, 
)

[I 2026-06-14 16:06:01,120] Using an existing study with name 'xgb_bike' instead of creating a new one.


In [14]:
# study.optimize(objective, n_trials=600)

In [15]:
print(f"[XGBoost + optuna]\nbest_params={study.best_params}\nbest_value={study.best_value}")

[XGBoost + optuna]
best_params={'max_depth': 4, 'n_estimators': 327, 'learning_rate': 0.03238187185153745, 'subsample': 0.6529338262062644, 'colsample_bytree': 0.5404925825289562, 'colsample_bylevel': 0.6715431155339864, 'colsample_bynode': 0.7691574932488607, 'min_child_weight': 5, 'gamma': 0.003830751466774046, 'max_delta_step': 2.684061914432476, 'reg_alpha': 0.1561300464404131, 'reg_lambda': 0.34956855822159777}
best_value=515.5047389833941


In [24]:
best_params = study.best_params

best_params={'max_depth': 6, 'n_estimators': 420, 'learning_rate': 0.03013096996514685, 'subsample': 0.6078309671571543, 'colsample_bytree': 0.5462451377972369, 'colsample_bylevel': 0.7603391791039791, 'colsample_bynode': 0.5873296797466079, 'min_child_weight': 5, 'gamma': 0.0005695450524774944, 'max_delta_step': 1.3001732261195198, 'reg_alpha': 0.13954346021294947, 'reg_lambda': 3.877662825916745, 'tree_method': 'hist', 'random_state': 42}

best_params = {**best_params, "tree_method": "hist", "random_state": RANDOM_STATE,}

### Prediction

In [25]:
val_rmses = []
for train_idx, val_idx in kf.split(X, y_log, groups=groups):
    model = xgb.XGBRegressor(
        **best_params
    )

    model.fit(X[train_idx], y_log[train_idx])
    val_pred = np.expm1(model.predict(X[val_idx]))
    y_val_orig = np.expm1(y_log[val_idx])
    val_rmses.append(rmse(y_val_orig, val_pred))

print(f"[XGBoost + optuna]\nbest_params={best_params}\nbest_value={float(np.mean(val_rmses))}")

[XGBoost + optuna]
best_params={'max_depth': 6, 'n_estimators': 420, 'learning_rate': 0.03013096996514685, 'subsample': 0.6078309671571543, 'colsample_bytree': 0.5462451377972369, 'colsample_bylevel': 0.7603391791039791, 'colsample_bynode': 0.5873296797466079, 'min_child_weight': 5, 'gamma': 0.0005695450524774944, 'max_delta_step': 1.3001732261195198, 'reg_alpha': 0.13954346021294947, 'reg_lambda': 3.877662825916745, 'tree_method': 'hist', 'random_state': 42}
best_value=515.4421873801151


In [18]:
model = xgb.XGBRegressor(**best_params)

In [19]:
model.fit(X, y_log)

pred = np.expm1(model.predict(X_test))

# Submission Generator

In [20]:
from scripts.save_submission import save_submission_csv

In [21]:
df_date = pd.read_csv(f"data-00-raw/bike_test.csv", parse_dates=["date"])

df_date["date"]

0     2023-01-01
1     2023-01-02
2     2023-01-03
3     2023-01-04
4     2023-01-05
         ...    
360   2023-12-27
361   2023-12-28
362   2023-12-29
363   2023-12-30
364   2023-12-31
Name: date, Length: 365, dtype: datetime64[us]

In [22]:
df_submission = pd.DataFrame({"date": df_date["date"], "pred": pred})

df_submission

,date,pred
0,2023-01-01,1222.920654
1,2023-01-02,1614.320312
2,2023-01-03,1273.718628
3,2023-01-04,1320.582886
4,2023-01-05,1775.226440
...,...,...
360,2023-12-27,776.946838
361,2023-12-28,1493.681030
362,2023-12-29,1955.901733
363,2023-12-30,1430.759277


In [23]:
save_submission_csv(df_submission)